# 🩺 Transformer + Explainable AI Based Cuffless Blood Pressure Estimation using PPG

---

## 📌 Project Objective

The objective of this project is to estimate a person's blood pressure without using a traditional inflatable cuff.

Instead of measuring blood pressure directly, we use a physiological signal called the **Photoplethysmogram (PPG)**.

The PPG signal is processed using signal processing techniques and converted into scalogram images.

These images are then given to a Transformer-based Deep Learning model that predicts:

- Systolic Blood Pressure (SBP)
- Diastolic Blood Pressure (DBP)

Finally, Explainable AI (XAI) techniques are used to understand how the model makes predictions.

---

## Workflow

MIMIC-IV Dataset

↓

Extract PPG

↓

Preprocessing

↓

Filtering

↓

Heart Rate

↓

Continuous Wavelet Transform

↓

Scalogram

↓

Vision Transformer

↓

Blood Pressure Prediction

↓

Explainable AI

↓

Evaluation

---

## Technologies Used

- Python
- NumPy
- Pandas
- Matplotlib
- SciPy
- WFDB
- PyTorch
- timm
- Captum
- SHAP


In [ ]:
import os
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy import signal

import wfdb

import pywt

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")

# Setting Random Seed

Random seed ensures that our experiment gives the same results every time we run the notebook.

In [ ]:
# =====================================
# Project Configuration
# =====================================

PROJECT_NAME = "Transformer + XAI Cuffless BP Estimation"

RANDOM_SEED = 42

DATASET_VERSION = "MIMIC-IV Waveform Database 0.1.0"

PPG_CHANNEL_NAMES = [
    "PLETH",
    "PPG"
]

ABP_CHANNEL_NAMES = [
    "ABP",
    "ART"
]

OUTPUT_IMAGE_SIZE = (224, 224)

print(PROJECT_NAME)
print(DATASET_VERSION)

In [ ]:
SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random Seed Set Successfully!")

In [ ]:
folders = [
    "data/raw",
    "data/processed",
    "data/labels",
    "models",
    "outputs",
    "outputs/figures",
    "outputs/csv",
    "outputs/logs",
    "scalograms",
    "checkpoints",
    "notebooks"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✅ Project folders created successfully!")

# Section 7: Configure Dataset Path

In this section, we define the location of the MIMIC-IV Waveform Dataset.

Instead of writing the dataset path repeatedly throughout the notebook, we store it in one variable. This makes the notebook easier to maintain and allows the dataset location to be changed in a single place if needed.

In [ ]:
from pathlib import Path

DATASET_PATH = Path(
    r"C:\Users\diksh\OneDrive\Documents\MIMIC4\MIMIC4\physionet.org\files\mimic4wdb"
)

print(DATASET_PATH)
print("Exists :", DATASET_PATH.exists())

In [ ]:
import os

print("=" * 70)
print("MIMIC-IV Waveform Dataset")
print("=" * 70)

for item in sorted(os.listdir(DATASET_PATH)):
    print(item)

In [ ]:
from pathlib import Path

root = Path(DATASET_PATH)

print("=" * 70)
print("MIMIC-IV Folder Structure")
print("=" * 70)

for item in root.iterdir():
    print(item.name)

    if item.is_dir():
        print("--------------------------------------------------")

        count = 0

        for sub in item.iterdir():
            print("   └──", sub.name)

            count += 1

            if count >= 10:
                print("   ...")
                break

# Section 8 : Reading the RECORDS File

The MIMIC-IV Waveform Dataset contains thousands of waveform records.

Instead of manually searching every folder, the RECORDS file contains the relative path of every available waveform record.

In this section we will:

- Read the RECORDS file.
- Count the total number of waveform records.
- Display the first few records.

In [ ]:
# ==========================================
# Read RECORDS File
# ==========================================

RECORDS_FILE = DATASET_PATH / "0.1.0" / "RECORDS"

print("RECORDS File:")
print(RECORDS_FILE)

print("\nExists :", RECORDS_FILE.exists())

In [ ]:
# Read all waveform record paths

with open(RECORDS_FILE, "r") as file:
    record_paths = file.read().splitlines()

print("=" * 70)
print("Total Waveform Records")
print("=" * 70)

print(len(record_paths))

In [ ]:
print("=" * 70)
print("First 20 Records")
print("=" * 70)

for i, record in enumerate(record_paths[:20], start=1):
    print(f"{i:02d}. {record}")

# Section 9 : Finding WFDB Records

The RECORDS file only contains folder names.

Each folder may contain one or more WFDB records.

A WFDB record is usually composed of:

- .hea  (Header)
- .dat  (Signal)
- Optional annotation files

In this section we will:

1. Scan every record folder.
2. Find all WFDB header (.hea) files.
3. Build a list of actual waveform records.
4. Count the number of usable records.

In [ ]:
from pathlib import Path

# Root folder containing waveform records
WAVE_ROOT = DATASET_PATH / "0.1.0"

print("Wave Root:")
print(WAVE_ROOT)

print("\nExists :", WAVE_ROOT.exists())

In [ ]:
import os

wfdb_records = []

for folder in record_paths:

    folder_path = WAVE_ROOT / folder

    if not folder_path.exists():
        continue

    for file in os.listdir(folder_path):

        if file.endswith(".hea"):

            record_name = file.replace(".hea", "")

            wfdb_records.append(folder_path / record_name)

print("=" * 70)
print("Total WFDB Records Found")
print("=" * 70)

print(len(wfdb_records))

In [ ]:
print("=" * 70)
print("First 10 WFDB Records")
print("=" * 70)

for record in wfdb_records[:10]:
    print(record)

In [ ]:
from pathlib import Path

first_folder = WAVE_ROOT / record_paths[0]

print("First Folder:")
print(first_folder)

print("\nContents:\n")

for item in first_folder.iterdir():
    print(item.name)
    

In [ ]:
from pathlib import Path

patient_folder = None

for item in first_folder.iterdir():

    if item.is_dir():

        patient_folder = item
        break

print(patient_folder)

In [ ]:
print("=" * 70)
print("Files inside Patient Folder")
print("=" * 70)

for file in sorted(patient_folder.iterdir()):
    print(file.name)

# Section 10 : Read First WFDB Header

Each patient folder contains one master header file.

The master header contains:

- Sampling Frequency
- Number of Segments
- Signal Information

We first inspect this header before reading waveform signals.

In [ ]:
from pathlib import Path

# Find the master .hea file (ignore segmented files like _0001.hea)
master_header = None

for file in patient_folder.glob("*.hea"):
    if "_" not in file.stem:
        master_header = file
        break

print("=" * 70)
print("Master Header File")
print("=" * 70)

print(master_header)

In [ ]:
# Remove the .hea extension because WFDB expects only the record name
master_record = str(master_header.with_suffix(""))

print(master_record)

In [ ]:
import wfdb

header = wfdb.rdheader(master_record)

print(header)

In [ ]:
print("=" * 70)
print("Sampling Frequency")
print("=" * 70)

print(header.fs)

In [ ]:
print("=" * 70)
print("Signal Names")
print("=" * 70)

print(header.sig_name)

In [ ]:
print("=" * 70)
print("Record Name")
print("=" * 70)

print(header.record_name)

In [ ]:
print("=" * 70)
print("Number of Segments")
print("=" * 70)

print(header.n_seg)

In [ ]:
print("=" * 70)
print("Segment Names")
print("=" * 70)

print(header.seg_name)

In [ ]:
print("=" * 70)
print("Segment Lengths")
print("=" * 70)

print(header.seg_len)

# Section 11 : Reading the First Segment

The master record only stores metadata.

Each segment contains the actual physiological signals.

In this section we will:

- Read the first segment.
- Display all available channels.
- Read waveform values.
- Display signal dimensions.

In [ ]:
# ===========================================
# Read First Segment
# ===========================================

segment_name = header.seg_name[0]

print("First Segment :", segment_name)

In [ ]:
segment_record = patient_folder / segment_name

print(segment_record)

In [ ]:
print("=" * 70)
print("Searching First Valid Segment")
print("=" * 70)

valid_segment = None

for seg_name, seg_len in zip(header.seg_name, header.seg_len):

    if seg_len > 0:
        valid_segment = seg_name
        print("Found :", seg_name)
        print("Length:", seg_len)
        break

# Section 12 : Automatic Dataset Scanner

This section scans the complete MIMIC-IV dataset.

Steps

1. Read every patient
2. Read every segment
3. Ignore empty segments
4. Detect PPG
5. Detect ABP
6. Store metadata

In [ ]:
import pandas as pd
import wfdb
from pathlib import Path
from tqdm import tqdm

metadata = []

In [ ]:
def find_master_header(patient_folder):
    """
    Finds the master WFDB header inside a patient folder.
    """

    for file in patient_folder.glob("*.hea"):

        if "_" not in file.stem:

            return file.with_suffix("")

    return None

In [ ]:
wave_root = DATASET_PATH / "0.1.0"

print(wave_root)

In [ ]:
patients = []

for record in record_paths:

    folder = wave_root / record

    if folder.exists():

        patients.append(folder)

print("Patients Found :", len(patients))

In [ ]:
patients[:5]

In [ ]:
from pathlib import Path
import wfdb
from tqdm import tqdm

metadata = []

for patient_path in tqdm(record_paths):

    patient_folder = WAVE_ROOT / patient_path

    if not patient_folder.exists():
        continue

    # Every patient folder contains subfolders like 81739927
    for record_folder in patient_folder.iterdir():

        if not record_folder.is_dir():
            continue

        master_header = record_folder / f"{record_folder.name}.hea"

        if not master_header.exists():
            continue

        try:
            header = wfdb.rdheader(str(master_header.with_suffix("")))
        except Exception:
            continue

        for seg_name, seg_len in zip(header.seg_name, header.seg_len):

            if seg_len == 0:
                continue

            segment_file = record_folder / seg_name

            try:
                record = wfdb.rdrecord(str(segment_file))
            except Exception:
                continue

            channels = record.sig_name

            has_ppg = any(ch.upper() in ["PLETH", "PPG"] for ch in channels)
            has_abp = any(ch.upper() in ["ABP", "ART"] for ch in channels)

            metadata.append({
                "patient": patient_folder.name,
                "record": record_folder.name,
                "segment": seg_name,
                "length": seg_len,
                "fs": record.fs,
                "channels": ",".join(channels),
                "has_ppg": has_ppg,
                "has_abp": has_abp
            })
print("Collected records:", len(metadata))

In [ ]:
import pandas as pd

metadata = pd.DataFrame(metadata)

print("="*70)
print("Metadata Shape")
print("="*70)

print(metadata.shape)

print("\nColumns:\n")
print(metadata.columns.tolist())

metadata.head()

In [ ]:
metadata.to_csv(
    "data/labels/metadata.csv",
    index=False
)

print("metadata.csv saved successfully.")

In [ ]:
print(metadata["channels"].value_counts().head(20))

In [ ]:
print(metadata["has_ppg"].value_counts())

In [ ]:
print(metadata["has_abp"].value_counts())

In [ ]:
valid_segments = metadata[
    metadata["has_ppg"] &
    metadata["has_abp"]
]

print(valid_segments.shape)

valid_segments.head()

In [ ]:
import wfdb
import numpy as np
from tqdm import tqdm

# ============================================
# Extract SBP and DBP from ABP channel
# ============================================

sbp_list = []
dbp_list = []

missing_record = 0
missing_abp = 0
empty_abp = 0
good = 0

for _, row in tqdm(valid_segments.iterrows(), total=len(valid_segments)):

    patient = row["patient"]
    record = row["record"]
    segment = row["segment"]

    record_path = (
        WAVE_ROOT /
        f"waves/{patient[:4]}/{patient}" /
        record /
        segment
    )

    # -------------------------------
    # Read record
    # -------------------------------
    try:
        rec = wfdb.rdrecord(str(record_path))
    except Exception:
        missing_record += 1
        sbp_list.append(np.nan)
        dbp_list.append(np.nan)
        continue

    channels = [c.upper() for c in rec.sig_name]

    # -------------------------------
    # Check ABP channel
    # -------------------------------
    if "ABP" not in channels:
        missing_abp += 1
        sbp_list.append(np.nan)
        dbp_list.append(np.nan)
        continue

    abp = rec.p_signal[:, channels.index("ABP")]

    # -------------------------------
    # Remove NaN / Inf values
    # -------------------------------
    abp = abp[np.isfinite(abp)]

    # -------------------------------
    # Keep physiologically valid BP
    # -------------------------------
    abp = abp[(abp > 20) & (abp < 250)]

    if len(abp) == 0:
        empty_abp += 1
        sbp_list.append(np.nan)
        dbp_list.append(np.nan)
        continue

    # -------------------------------
    # Robust BP estimation
    # -------------------------------
    sbp = np.percentile(abp, 95)
    dbp = np.percentile(abp, 5)

    sbp_list.append(sbp)
    dbp_list.append(dbp)

    good += 1

# ============================================
# Summary
# ============================================

print("=" * 60)
print("Good records      :", good)
print("Missing record    :", missing_record)
print("Missing ABP       :", missing_abp)
print("Empty ABP         :", empty_abp)
print("=" * 60)

In [ ]:
valid_segments["SBP"] = sbp_list
valid_segments["DBP"] = dbp_list

print(valid_segments.shape)

valid_segments.head()

In [ ]:
valid_segments = valid_segments.dropna()

print(valid_segments.shape)

valid_segments[["SBP","DBP"]].describe()

In [ ]:
print(metadata.shape)

metadata.head()

In [ ]:
metadata[
    (metadata["has_ppg"] == True) &
    (metadata["has_abp"] == True)
].shape

In [ ]:
valid_segments = metadata[
    (metadata["has_ppg"] == True) &
    (metadata["has_abp"] == True)
].copy()

print(valid_segments.shape)

In [ ]:
print("Metadata rows :", len(metadata))
print("Valid segments:", len(valid_segments))

In [ ]:
print(metadata["has_abp"].value_counts())
print(metadata["has_ppg"].value_counts())

# Section 4 : Visualize Five Random Subjects

In [ ]:
import random

random.seed(42)

five_subjects = random.sample(
    valid_segments["record"].unique().tolist(),
    5
)

print(five_subjects)

In [ ]:
five_subjects

In [ ]:
import wfdb
import matplotlib.pyplot as plt

In [ ]:
for record_id in five_subjects:

    row = valid_segments[
        valid_segments["record"] == record_id
    ].iloc[0]

    patient = row["patient"]
    segment = row["segment"]

    record_path = (
        WAVE_ROOT
        / f"waves/{patient[:4]}/{patient}"
        / record_id
        / segment
    )

    record = wfdb.rdrecord(str(record_path))

    channels = record.sig_name

    ppg_index = channels.index("Pleth")

    ppg = record.p_signal[:, ppg_index]

    plt.figure(figsize=(15,4))

    plt.plot(ppg)

    plt.title(f"{patient} | {record_id}")

    plt.xlabel("Samples")

    plt.ylabel("Amplitude")

    plt.show()

In [ ]:
import numpy as np

valid_segments = valid_segments.copy()

valid_segments = valid_segments[
    (valid_segments["length"] >= valid_segments["fs"] * 60)
]

print(valid_segments.shape)

In [ ]:
import matplotlib.pyplot as plt
import wfdb
import random

random.seed(42)

sample_rows = valid_segments.sample(5)

for _, row in sample_rows.iterrows():

    patient = row["patient"]
    record = row["record"]
    segment = row["segment"]

    record_path = (
        WAVE_ROOT /
        f"waves/{patient[:4]}/{patient}" /
        record /
        segment
    )

    rec = wfdb.rdrecord(str(record_path))

    ppg_index = rec.sig_name.index("Pleth")

    ppg = rec.p_signal[:, ppg_index]

    fs = rec.fs

    # Show only the first 10 seconds
    ppg = ppg[: int(fs * 10)]

    plt.figure(figsize=(12,4))
    plt.plot(ppg)
    plt.title(f"{patient} | {record}")
    plt.xlabel("Samples")
    plt.ylabel("PPG")
    plt.show()

In [ ]:
from scipy.signal import butter, filtfilt

def butter_bandpass(signal, fs):

    low = 0.5
    high = 8

    nyquist = fs / 2

    b, a = butter(
        4,
        [low / nyquist, high / nyquist],
        btype="band"
    )

    return filtfilt(b, a, signal)

In [ ]:
filtered = butter_bandpass(ppg, fs)

plt.figure(figsize=(15,5))

plt.plot(ppg, label="Raw")

plt.plot(filtered, label="Filtered")

plt.legend()

plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

filtered = scaler.fit_transform(
    filtered.reshape(-1,1)
).flatten()

plt.figure(figsize=(15,4))
plt.plot(filtered)
plt.title("Normalized PPG")
plt.show()

In [ ]:
from scipy.signal import butter, filtfilt
from sklearn.preprocessing import MinMaxScaler

def preprocess_ppg(ppg, fs):

    b,a = butter(
        4,
        [0.5/(fs/2),8/(fs/2)],
        btype="band"
    )

    filtered = filtfilt(b,a,ppg)

    scaler = MinMaxScaler()

    normalized = scaler.fit_transform(
        filtered.reshape(-1,1)
    ).flatten()

    return normalized

In [ ]:
processed = preprocess_ppg(ppg, fs)

plt.figure(figsize=(15,4))
plt.plot(processed)
plt.title("Processed PPG")
plt.show()

In [ ]:
import os

OUTPUT_DIR = "scalograms_final"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output folder ready!")

In [ ]:
import shutil
import os

if os.path.exists("scalograms_final"):
    shutil.rmtree("scalograms_final")

os.makedirs("scalograms_final")

print("Folder cleaned.")

In [ ]:
import os
import cv2
import pywt
import wfdb
import numpy as np
import pandas as pd
from tqdm import tqdm

OUTPUT_DIR = "scalograms_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_SECONDS = 10
MAX_WINDOWS = 6

all_labels = []
saved = 0

for _, row in tqdm(valid_segments.iterrows(), total=len(valid_segments)):

    patient = row["patient"]
    record = row["record"]
    segment = row["segment"]

    record_path = (
        WAVE_ROOT /
        f"waves/{patient[:4]}/{patient}" /
        record /
        segment
    )

    try:
        rec = wfdb.rdrecord(str(record_path))
    except:
        continue

    channels = [c.upper() for c in rec.sig_name]

    if "PLETH" in channels:
        idx = channels.index("PLETH")
    elif "PPG" in channels:
        idx = channels.index("PPG")
    else:
        continue

    ppg = rec.p_signal[:, idx]

    if len(ppg) < 30:
        continue

    try:
        processed = preprocess_ppg(ppg, rec.fs)
    except Exception:
        continue

   

    WINDOW = int(WINDOW_SECONDS * rec.fs)

    if len(processed) < WINDOW:
        continue

    num_windows = min(
        MAX_WINDOWS,
        len(processed) // WINDOW
    )

    for i in range(num_windows):

        start = i * WINDOW
        end = start + WINDOW

        window = processed[start:end]

        if len(window) != WINDOW:
            continue

        coef, _ = pywt.cwt(
            window,
            np.arange(1,129),
            "morl"
        )

        img = np.abs(coef)

        img -= img.min()

        if img.max() > 0:
            img /= img.max()

        img = (img * 255).astype(np.uint8)

        img = cv2.applyColorMap(
            img,
            cv2.COLORMAP_JET
        )

        img = cv2.resize(
            img,
            (224,224),
            interpolation=cv2.INTER_AREA
        )

        filename = (
            f"{patient}_{record}_{segment}_{i}.png"
        )

        save_path = os.path.join(
            OUTPUT_DIR,
            filename
        )

        cv2.imwrite(save_path, img)

        all_labels.append({
            "image": filename,
            "SBP": row["SBP"],
            "DBP": row["DBP"]
        })

        saved += 1

labels_final = pd.DataFrame(all_labels)

labels_final.to_csv(
    "labels_final.csv",
    index=False
)

print("=" * 60)
print("Images Generated :", saved)
print("Labels Saved     :", len(labels_final))
print("=" * 60)

In [ ]:
import os
import pandas as pd

labels_final = pd.read_csv("labels_final.csv")

print("="*60)
print("Images :", len(os.listdir("scalograms_final")))
print("Labels :", len(labels_final))
print("="*60)

labels_final.head()

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

files = sorted(os.listdir("scalograms_final"))[:6]

plt.figure(figsize=(15,6))

for i, file in enumerate(files):

    img = cv2.imread(
        os.path.join("scalograms_final", file)
    )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    plt.subplot(2,3,i+1)
    plt.imshow(img)
    plt.title(file[-8:])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import os
import pandas as pd

labels = pd.read_csv("labels_final.csv")

print("="*60)
print("Images :", len(os.listdir("scalograms_final")))
print("Labels :", len(labels))
print("="*60)

labels.head()

In [ ]:
print(labels.describe())

print("SBP < 50 :", (labels["SBP"] < 50).sum())
print("DBP < 30 :", (labels["DBP"] < 30).sum())